# LSTM 做图像分类
前面我们讲了 LSTM 特别适合做序列类型的数据，那么 LSTM 能不能想 CNN 一样用来做图像分类呢？下面我们用 MNIST 手写字体的例子来展示一下如何用 LSTM 做图像分类，但是这种方法并不是主流，这里我们只是作为举例。

对于一张手写字体的图片，其大小是 28 * 28，我们可以将其看做是一个长为 28 的序列，每个序列的特征都是 28，也就是

![MNIST](images/rnn_mnist_classification.jpeg)

这样我们解决了输入序列的问题，对于输出序列怎么办呢？其实非常简单，虽然我们的输出是一个序列，但是我们只需要保留其中一个作为输出结果就可以了，这样的话肯定保留最后一个结果是最好的，因为最后一个结果有前面所有序列的信息。

下面我们直接通过例子展示

In [2]:
import torch
from torch.autograd import Variable
from torch import nn
from torch.utils.data import DataLoader

from torchvision import transforms as tfs
from torchvision.datasets import MNIST

In [3]:
# 定义数据
data_tf = tfs.Compose([
    tfs.ToTensor(),
    tfs.Normalize([0.5], [0.5]) # 标准化
])

train_set = MNIST('../../data/mnist', train=True, transform=data_tf)
test_set  = MNIST('../../data/mnist', train=False, transform=data_tf)

train_data = DataLoader(train_set, 64, True,  num_workers=4)
test_data  = DataLoader(test_set, 128, False, num_workers=4)

In [4]:
# 定义LSTM模型
class LSTM_Classify(nn.Module):
    def __init__(self, in_feature=28, hidden_feature=100, num_class=10, num_layers=2):
        super(LSTM_Classify, self).__init__()
        self.rnn = nn.LSTM(in_feature, hidden_feature, num_layers) # 使用两层 LSTM
        self.classifier = nn.Linear(hidden_feature, num_class) # 将最后一个 rnn 的输出使用全连接得到最后的分类结果
        
    def forward(self, x):
        '''
        x 大小为 (batch, 1, 28, 28)，所以我们需要将其转换成 RNN 的输入形式，即 (28, batch, 28)
        '''
        x = x.squeeze() # 去掉 (batch, 1, 28, 28) 中的 1，变成 (batch, 28, 28)
        x = x.permute(2, 0, 1) # 将最后一维放到第一维，变成 (28, batch, 28)
        out, _ = self.rnn(x) # 使用默认的隐藏状态，得到的 out 是 (28, batch, hidden_feature)
        out = out[-1, :, :] # 取序列中的最后一个，大小是 (batch, hidden_feature)
        out = self.classifier(out) # 得到分类结果
        return out

In [5]:
lstm = LSTM_Classify()
criterion = nn.CrossEntropyLoss()

optimzier = torch.optim.Adam(lstm.parameters(), 1e-2)

In [6]:
# 开始训练
from utils import train
train(lstm, train_data, test_data, 10, optimzier, criterion)

Epoch 0. Train Loss: 0.425133, Train Acc: 0.860974, Valid Loss: 0.144821, Valid Acc: 0.957872, Time 00:00:04
Epoch 1. Train Loss: 0.132112, Train Acc: 0.962670, Valid Loss: 0.142214, Valid Acc: 0.958070, Time 00:00:04
Epoch 2. Train Loss: 0.107389, Train Acc: 0.968184, Valid Loss: 0.124949, Valid Acc: 0.961630, Time 00:00:05
Epoch 3. Train Loss: 0.102706, Train Acc: 0.969866, Valid Loss: 0.088188, Valid Acc: 0.973497, Time 00:00:04
Epoch 4. Train Loss: 0.097741, Train Acc: 0.970782, Valid Loss: 0.083919, Valid Acc: 0.974486, Time 00:00:04
Epoch 5. Train Loss: 0.103547, Train Acc: 0.969533, Valid Loss: 0.104923, Valid Acc: 0.968354, Time 00:00:04
Epoch 6. Train Loss: 0.104536, Train Acc: 0.969266, Valid Loss: 0.104912, Valid Acc: 0.970530, Time 00:00:05
Epoch 7. Train Loss: 0.099223, Train Acc: 0.971015, Valid Loss: 0.134043, Valid Acc: 0.963113, Time 00:00:05
Epoch 8. Train Loss: 0.108463, Train Acc: 0.968134, Valid Loss: 0.101802, Valid Acc: 0.971025, Time 00:00:05
Epoch 9. Train Loss

可以看到，训练 10 次在简单的 mnist 数据集上也取得的了 96% 的准确率，所以说 RNN 也可以做做简单的图像分类，但是这并不是他的主战场，下次课我们会讲到 RNN 的一个使用场景，时间序列预测。

In [9]:
# 定义RNN模型
class RNN_Classify(nn.Module):
    def __init__(self, in_feature=28, hidden_feature=100, num_class=10, num_layers=2):
        super(RNN_Classify, self).__init__()
        self.rnn = nn.RNN(in_feature, hidden_feature, num_layers) # 使用两层 LSTM
        self.classifier = nn.Linear(hidden_feature, num_class) # 将最后一个 rnn 的输出使用全连接得到最后的分类结果
        
    def forward(self, x):
        '''
        x 大小为 (batch, 1, 28, 28)，所以我们需要将其转换成 RNN 的输入形式，即 (28, batch, 28)
        '''
        x = x.squeeze() # 去掉 (batch, 1, 28, 28) 中的 1，变成 (batch, 28, 28)
        x = x.permute(2, 0, 1) # 将最后一维放到第一维，变成 (28, batch, 28)
        out, _ = self.rnn(x) # 使用默认的隐藏状态，得到的 out 是 (28, batch, hidden_feature)
        out = out[-1, :, :] # 取序列中的最后一个，大小是 (batch, hidden_feature)
        out = self.classifier(out) # 得到分类结果
        return out

rnn = RNN_Classify()
criterion = nn.CrossEntropyLoss()

optimzier = torch.optim.Adam(rnn.parameters(), 1e-2)

# 开始训练
train(rnn, train_data, test_data, 10, optimzier, criterion)

Epoch 0. Train Loss: 2.375556, Train Acc: 0.101629, Valid Loss: 2.377000, Valid Acc: 0.101266, Time 00:00:03
Epoch 1. Train Loss: 2.370965, Train Acc: 0.102229, Valid Loss: 2.404703, Valid Acc: 0.113627, Time 00:00:03
Epoch 2. Train Loss: 2.381337, Train Acc: 0.101912, Valid Loss: 2.352402, Valid Acc: 0.113627, Time 00:00:03
Epoch 3. Train Loss: 2.368066, Train Acc: 0.100580, Valid Loss: 2.371308, Valid Acc: 0.100475, Time 00:00:03
Epoch 4. Train Loss: 2.374764, Train Acc: 0.100996, Valid Loss: 2.396465, Valid Acc: 0.102354, Time 00:00:04
Epoch 5. Train Loss: 2.373966, Train Acc: 0.102779, Valid Loss: 2.425117, Valid Acc: 0.097013, Time 00:00:04
Epoch 6. Train Loss: 2.373584, Train Acc: 0.103378, Valid Loss: 2.449691, Valid Acc: 0.103441, Time 00:00:04
Epoch 7. Train Loss: 2.374890, Train Acc: 0.100613, Valid Loss: 2.365471, Valid Acc: 0.096123, Time 00:00:04
Epoch 8. Train Loss: 2.371989, Train Acc: 0.100130, Valid Loss: 2.453573, Valid Acc: 0.097607, Time 00:00:04
Epoch 9. Train Loss